# Habitat suitability under climate change

Our changing climate is changing where plant species can live,
and conservation and restoration practices will need to take
this into
account.

In this coding challenge, you will create a habitat suitability model
for a terrestrial plant species of your choice that lives in the contiguous United States
(CONUS). We have this limitation because the downscaled climate data we
suggest, the [MACAv2 dataset](https://www.climatologylab.org/maca.html),
is only available in the CONUS – if you find other downscaled climate
data at an appropriate resolution, you are welcome to choose a different
study area. If you don’t have anything in mind, you can take a look at
[*Sorghastrum nutans*](https://www.gbif.org/species/2704414), a grass native to North America. In the past 50
years, its range has moved
northward.

Your suitability assessment will be based on combining multiple data
layers related to soil, topography, and climate, then applying a fuzzy logic model across the different data layers to generate habitat suitability maps. 

You will need to create a **modular, reproducible, workflow** using functions and loops.
To do this effectively, we recommend planning your code out in advance
using a technique such as a pseudocode outline or a flow diagram. We
recommend breaking each of the blocks below out into multiple steps. It
is unnecessary to write a step for every line of code unless you find
that useful. As a rule of thumb, aim for steps that cover the major
structures of your code in 2-5 line chunks.

In [1]:
### Import libraries
# File paths
import os
import pathlib 
from glob import glob
from pathlib import Path

# Work with GBIF data
import pygbif.occurrences as occ
import pygbif.species as species
from getpass import getpass

# Unzipping
import zipfile 
import time

# Work with spatial data
import geopandas as gpd 
import xrspatial

# Work with dataframes
import pandas as pd
import numpy as np

# Plot and visualize
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import panel as pn
import cartopy.crs as ccrs
import rioxarray as rxr
import rioxarray.merge as rxrm

# Deal with invalid geometries
from shapely.geometry import MultiPolygon, Polygon

# for API use
import requests

# Search for locations by name - this might take a moment
from osmnx import features as osm

In [2]:
### Set up file paths
# Set up directory
data_dir = os.path.join(

    pathlib.Path.home(),

    # Earth Analytics
    'Documents',
    'education',
    'earth-data-analytics',
    'spring-2026-data',

    # Create project directory
    'douglas-fir-habitat-suitability'
)

# Make the dir once
os.makedirs(data_dir, exist_ok=True)

In [3]:
# Set a dir for the gbif data
gbif_dir = os.path.join(data_dir, 'gbif_douglas_fir')

In [4]:
### Login to GBIF account and do not save credentials
# Reset credentials
reset_credentials = False

# Request and store username
if (not ('GBIF_USER'  in os.environ)) or reset_credentials:
    os.environ['GBIF_USER'] = input('GBIF username:')

# Securely request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset_credentials:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
# Request and store account email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset_credentials:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [5]:
'GBIF_PWD' in os.environ

True

In [6]:
# Set species name
species_name = "Pseudotsuga menziesii var. menziesii"

# Species info from GBIF
species_info = species.name_lookup(species_name,
                                   rank = 'Species')

# Grab the first result
first_result = species_info['results'][0]
first_result

{'key': 100326733,
 'datasetKey': '16c3f9cb-4b19-4553-ac8e-ebb90003aa02',
 'nubKey': 2685796,
 'parentKey': 314611296,
 'parent': 'Pseudotsuga',
 'kingdom': 'Plantae',
 'order': 'Coniferales',
 'family': 'Pinaceae',
 'genus': 'Pseudotsuga',
 'species': 'Pseudotsuga menziesii',
 'kingdomKey': 314608991,
 'classKey': 165185887,
 'orderKey': 314611246,
 'familyKey': 314611295,
 'genusKey': 314611296,
 'speciesKey': 100326733,
 'scientificName': 'Pseudotsuga menziesii (Mirbel) Franco',
 'canonicalName': 'Pseudotsuga menziesii',
 'authorship': '(Mirbel) Franco',
 'nameType': 'SCIENTIFIC',
 'taxonomicStatus': 'ACCEPTED',
 'rank': 'SPECIES',
 'origin': 'SOURCE',
 'numDescendants': 0,
 'numOccurrences': 0,
 'taxonID': '360682',
 'habitats': [],
 'nomenclaturalStatus': [],
 'threatStatuses': [],
 'descriptions': [{'description': 'Die Gewöhnliche Douglasie (Pseudotsuga menziesii), oft einfach nur Douglasie oder umgangssprachlich auch Douglastanne, Douglasfichte, Douglaskiefer bzw. nach der Herku

In [7]:
# Get the species key
species_key = first_result['nubKey']

# Check that
first_result['species'], species_key

# Assign the species code
species_key = 2685796

In [8]:
# Make the file path
gbif_pattern = os.path.join(gbif_dir, '*.csv')

# Download it once
if not glob(gbif_pattern):

    # Submit the query
    gbif_query = occ.download([
        f"speciesKey = {species_key}",
        "hasCoordinate = True",
    ])

    # Only download once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']
        time.sleep(5)

    # Download the data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = data_dir
    )

    # Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)
# Find csv file path
gbif_path = glob(gbif_pattern)[0]

In [9]:
occ.download_meta("0000077-260221153910048")

{'key': '0000077-260221153910048',
 'doi': '10.15468/dl.hbkan4',
 'license': 'http://creativecommons.org/licenses/by-nc/4.0/legalcode',
 'request': {'predicate': {'type': 'and',
   'predicates': [{'type': 'equals',
     'key': 'SPECIES_KEY',
     'value': '2685796',
     'matchCase': False},
    {'type': 'equals',
     'key': 'HAS_COORDINATE',
     'value': 'True',
     'matchCase': False}]},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-02-21T16:26:37.587+00:00',
 'modified': '2026-02-21T16:48:01.648+00:00',
 'eraseAfter': '2026-08-21T16:26:37.481+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0000077-260221153910048.zip',
 'size': 14559947,
 'totalRecords': 141879,
 'numberDatasets': 951}

In [10]:
# Read csv and look at dataframe
gbif_df = pd.read_csv(
    gbif_path,
    delimiter = '\t'
)

# Check it out
gbif_df

C:\Users\nymve\AppData\Local\Temp\ipykernel_17656\2339774905.py:2: DtypeWarning: Columns (14,16,38,39,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  gbif_df = pd.read_csv(


,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,4051979777,017f23ba-bf04-4eb7-a076-04ab2e5de250,a20e0f9b-7634-459c-847c-ed9edceb56b3,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,Laurent LATHUILLIERE (ONF),NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:16.525Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...
1,4051549701,017f23ba-bf04-4eb7-a076-04ab2e5de250,80d93a33-8d6d-4d85-9310-931e441a4cbe,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,Laurent LATHUILLIERE (ONF),NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:36.946Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...
2,4051464820,017f23ba-bf04-4eb7-a076-04ab2e5de250,ac22faef-6696-473e-a7e9-38b1691e1028,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,Laurent LATHUILLIERE (ONF),NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:46.126Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...
3,4051297388,017f23ba-bf04-4eb7-a076-04ab2e5de250,5c3e4e4e-416d-41e4-8d78-71e1286f561e,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,Maud GIRONDE (ONF),NaN,CC_BY_4_0,NaN,Maud GIRONDE (ONF),NaN,NaN,2025-11-06T14:56:44.466Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...
4,4051769241,017f23ba-bf04-4eb7-a076-04ab2e5de250,dc7ce432-ab10-4df5-a467-f2715c1aa91b,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,Laurent LATHUILLIERE (ONF),NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:18.174Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141874,4529249471,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.7128.0.11398.7731,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:44.432Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
141875,4529069702,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.6813.0.4133.824,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:43.496Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
141876,4529184510,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.1014.128.1275.800,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:44.667Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
141877,4529246557,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.7128.0.11385.7718,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:44.653Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...


In [11]:
# Look at columns
gbif_df.columns


Index(['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'species', 'infraspecificEpithet',
       'taxonRank', 'scientificName', 'verbatimScientificName',
       'verbatimScientificNameAuthorship', 'countryCode', 'locality',
       'stateProvince', 'occurrenceStatus', 'individualCount',
       'publishingOrgKey', 'decimalLatitude', 'decimalLongitude',
       'coordinateUncertaintyInMeters', 'coordinatePrecision', 'elevation',
       'elevationAccuracy', 'depth', 'depthAccuracy', 'eventDate', 'day',
       'month', 'year', 'taxonKey', 'speciesKey', 'basisOfRecord',
       'institutionCode', 'collectionCode', 'catalogNumber', 'recordNumber',
       'identifiedBy', 'dateIdentified', 'license', 'rightsHolder',
       'recordedBy', 'typeStatus', 'establishmentMeans', 'lastInterpreted',
       'mediaType', 'issue'],
      dtype='object')

In [12]:
# Make it spatial
gbif_gdf = (
    gpd.GeoDataFrame(
        gbif_df,
        geometry = gpd.points_from_xy(
            gbif_df.decimalLongitude,
            gbif_df.decimalLatitude
        ),
        crs = 'EPSG:4326'
    )
)

# Check it out
gbif_gdf

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue,geometry
0,4051979777,017f23ba-bf04-4eb7-a076-04ab2e5de250,a20e0f9b-7634-459c-847c-ed9edceb56b3,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:16.525Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...,POINT (3.64187 45.80564)
1,4051549701,017f23ba-bf04-4eb7-a076-04ab2e5de250,80d93a33-8d6d-4d85-9310-931e441a4cbe,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:36.946Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...,POINT (3.02997 45.87488)
2,4051464820,017f23ba-bf04-4eb7-a076-04ab2e5de250,ac22faef-6696-473e-a7e9-38b1691e1028,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:46.126Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...,POINT (3.29359 45.63374)
3,4051297388,017f23ba-bf04-4eb7-a076-04ab2e5de250,5c3e4e4e-416d-41e4-8d78-71e1286f561e,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_4_0,NaN,Maud GIRONDE (ONF),NaN,NaN,2025-11-06T14:56:44.466Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...,POINT (7.23997 48.0063)
4,4051769241,017f23ba-bf04-4eb7-a076-04ab2e5de250,dc7ce432-ab10-4df5-a467-f2715c1aa91b,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_4_0,NaN,Laurent LATHUILLIERE (ONF),NaN,NaN,2025-11-06T14:56:18.174Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;FOOTPRINT_W...,POINT (3.02989 45.87413)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141874,4529249471,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.7128.0.11398.7731,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:44.432Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (2.57767 41.9562)
141875,4529069702,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.6813.0.4133.824,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:43.496Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (2.45702 41.9557)
141876,4529184510,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.1014.128.1275.800,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:44.667Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (2.21237 42.22447)
141877,4529246557,a4b7c71e-fc73-434e-a008-d63c222179d0,BDBC-C.7128.0.11385.7718,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,NaN,CC_BY_NC_4_0,Generalitat de Catalunya,NaN,NaN,NaN,2025-11-24T19:02:44.653Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,POINT (2.45778 41.86563)


In [13]:
# Plot it
gbif_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Douglas Fir occurrencces in GBIF',
    fill_color = None,
    line_color = 'purple',
    frame_width = 600
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]

### Load Sites

In [14]:
# Make directory for sites
site_dir = Path(data_dir) / "douglas_fir_sites"
site_dir.mkdir(parents=True, exist_ok=True)

In [15]:
# Create python path for zip file to live
zip_path = site_dir / "BdyAdm_LSRS_AdministrativeForest.zip"
print(zip_path)


C:\Users\nymve\Documents\education\earth-data-analytics\spring-2026-data\douglas-fir-habitat-suitability\douglas_fir_sites\BdyAdm_LSRS_AdministrativeForest.zip


In [16]:
# Unzip
# Create folder for the data
extract_folder = zip_path.parent

# Create the folder if it doesn't exist
extract_folder.mkdir(parents=True, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

In [ ]:
# Open the shapefile from forest boundaries
shp_path = extract_folder / "S_USA.BdyAdm_LSRS_AdministrativeForest.shp"

# Read the shapefile
gdf = gpd.read_file(shp_path)

# Check it out
print(gdf.head())
print(gdf.columns)

       ADMINFORES REGION FORESTNUMB FORESTORGC                     FORESTNAME  \
0  99040300010343     04         03       0403  Bridger-Teton National Forest   
1  99040700010343     04         07       0407          Dixie National Forest   
2  99040800010343     04         08       0408       Fishlake National Forest   
3  99041000010343     04         10       0410   Manti-La Sal National Forest   
4  99041200010343     04         12       0412        Payette National Forest   

     GIS_ACRES  SHAPE_AREA  SHAPE_LEN  \
0  3466359.302    1.554543  14.336621   
1  1711285.208    0.707542  11.739623   
2  1787975.173    0.748411  12.885034   
3  1414385.800    0.594270  11.093035   
4  2407306.442    1.113418  11.686712   

                                            geometry  
0  MULTIPOLYGON (((-109.98359 44.32437, -109.9832...  
1  MULTIPOLYGON (((-113.92462 37.67006, -113.9200...  
2  MULTIPOLYGON (((-112.1282 39.21301, -112.1282 ...  
3  MULTIPOLYGON (((-111.41206 39.9806, -111.41

In [18]:
# Filter for Willamette and Olympic
willamette_gdf = gdf[gdf['FORESTNAME'] == 'Willamette National Forest']
olympic_gdf = gdf[gdf['FORESTNAME'] == 'Olympic National Forest']

In [ ]:
# Check that they are geodataframes
type(willamette_gdf)
type(olympic_gdf)

geopandas.geodataframe.GeoDataFrame

In [ ]:
# Check out the forest
print(willamette_gdf)
print(olympic_gdf)

        ADMINFORES REGION FORESTNUMB FORESTORGC                  FORESTNAME  \
56  99061800010343     06         18       0618  Willamette National Forest   

      GIS_ACRES  SHAPE_AREA  SHAPE_LEN  \
56  1801291.974    0.819514   8.735849   

                                             geometry  
56  MULTIPOLYGON (((-122.2551 44.89989, -122.25464...  
        ADMINFORES REGION FORESTNUMB FORESTORGC               FORESTNAME  \
49  99060900010343     06         09       0609  Olympic National Forest   

     GIS_ACRES  SHAPE_AREA  SHAPE_LEN  \
49  697410.578    0.338175   8.807532   

                                             geometry  
49  MULTIPOLYGON (((-123.23506 48.01368, -123.2296...  


In [33]:
# Check the crs
willamette_gdf.crs
olympic_gdf.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [34]:
# Align crs to GBIF
willamette_gdf = willamette_gdf.to_crs(epsg = 4326)
olympic_gdf = olympic_gdf.to_crs(epsg = 4326)

In [37]:
# Check the polygon for Willamette
willamette_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Willamette National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [ ]:
# Check the polygon for Olympic
olympic_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Olympic National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [42]:
# Intersect with GBIF data with Willamette
douglas_willamette = gpd.overlay(gbif_gdf, willamette_gdf, how = 'intersection')

# Check out the occerences
douglas_willamette

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,issue,ADMINFORES,REGION,FORESTNUMB,FORESTORGC,FORESTNAME,GIS_ACRES,SHAPE_AREA,SHAPE_LEN,geometry
0,3820581763,b9f2645b-a5e0-4f24-aa97-eef1f6968fe8,{C302ACFC-1EAF-4A75-AE5A-E207FF9AB789},Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,GEODETIC_DATUM_ASSUMED_WGS84;CONTINENT_DERIVED...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.593 43.9653)
1,4061666597,e053ff53-c156-4e2e-b9b5-4462e9625424,urn:catalog:MO:Tropicos:100682193,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,OCCURRENCE_STATUS_INFERRED_FROM_INDIVIDUAL_COU...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.31861 43.845)
2,4929840492,f873ef66-231a-4ea3-bd4d-a16f182bf337,addbe8ee-51ba-413b-8355-093807e4d360,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,GEODETIC_DATUM_ASSUMED_WGS84;CONTINENT_DERIVED...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.31861 43.845)
3,4068929632,dfe29309-9f9f-493b-95e7-68f4563a3cdb,c787638c-bf40-441d-949b-289059cd1563,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,GEODETIC_DATUM_ASSUMED_WGS84;CONTINENT_DERIVED...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122 44)
4,2628924024,c13173f5-f372-4b1b-815f-2cd4f6f963a0,3f8cd8db7218a9c3416fcdff9411ca93,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,OCCURRENCE_STATUS_INFERRED_FROM_INDIVIDUAL_COU...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.593 43.9653)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,4978265708,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/24985...,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.13807 44.2778)
344,4952763852,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/24203...,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.11019 43.62486)
345,4046493383,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/71265903,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.23842 44.08302)
346,4851195097,7b742032-9940-4da3-8c1a-3eaceaed4f1f,fd44a58f-89e4-4b32-9673-08700666c1b3,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,CONTINENT_DERIVED_FROM_COORDINATES;INSTITUTION...,99061800010343,06,18,0618,Willamette National Forest,1801291.974,0.819514,8.735849,POINT (-122.17609 44.23375)


In [ ]:
# Intersect with GBIF data with Olympic
douglas_olympic = gpd.overlay(gbif_gdf, olympic_gdf, how = 'intersection')

# Check it out
douglas_olympic

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,issue,ADMINFORES,REGION,FORESTNUMB,FORESTORGC,FORESTNAME,GIS_ACRES,SHAPE_AREA,SHAPE_LEN,geometry
0,3016521033,24fd68c4-d4ac-4a39-8e8b-3af9dae81db4,1428000,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,CONTINENT_DERIVED_FROM_COORDINATES;INSTITUTION...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.87334 47.44988)
1,4851245862,7b742032-9940-4da3-8c1a-3eaceaed4f1f,bf6e4b98-1043-4782-a497-e924ffbecc34,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;GEODETIC_DATUM_ASSUMED_WGS8...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.41667 47.35)
2,3913299860,aea65dea-9908-41f7-bc9b-e9d18064ad28,24d18bb2-2051-11ea-9344-f75ddfefe904,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.13333 47.9)
3,3913300276,aea65dea-9908-41f7-bc9b-e9d18064ad28,2661c5fa-2051-11ea-9344-2fd26b4b0af3,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.13333 47.9)
4,3913302187,aea65dea-9908-41f7-bc9b-e9d18064ad28,2102b498-2051-11ea-9344-e7c1fb8db852,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.13333 47.9)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,3949504675,7a3679ef-5582-4aaa-81f0-8c2545cafc81,o-1007286016,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.15812 47.60342)
167,3949193006,7a3679ef-5582-4aaa-81f0-8c2545cafc81,o-1010881399,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.84157 47.46343)
168,5887323784,8310f570-f762-11e1-a439-00145eb45e9a,7b9992f9-524b-416e-9064-2ef4a951e47d,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,CONTINENT_DERIVED_FROM_COORDINATES;INSTITUTION...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.58146 48.06005)
169,5887457929,8310f570-f762-11e1-a439-00145eb45e9a,14b9f3cf-f210-44e7-b704-c6aa46044dca,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pseudotsuga,Pseudotsuga menziesii,...,GEODETIC_DATUM_ASSUMED_WGS84;CONTINENT_DERIVED...,99060900010343,06,09,0609,Olympic National Forest,697410.578,0.338175,8.807532,POINT (-123.0412 47.66503)


In [ ]:
# Check the occurences for Olympic
douglas_willamette.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Douglas Fir Occurances - Willamette National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]

In [ ]:
# Check the occurences for Olympic
douglas_olympic.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Douglas Fir Occurances - Olympic National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]

## STEP 1: Study overview

Before you begin coding, you will need to design your study.

### Step 1a: Select a species
Select the terrestrial plant species you want to study, and research its habitat parameters in scientific studies or other reliable sources. Individual studies may not have the breadth needed for this purpose, so take a look at reviews or overviews of the data. Do **not** just look at an AI-generated summary! In the US, the National Resource Conservation Service can have helpful fact sheets about different species. University Extension programs are also good resources for summaries.</p>
<p>Based on your research, select soil, topographic, and climate variables that you can use to determine if a particular location and time period is a suitable habitat for your species.</p></div></div>

**Reflect and respond**: 
Write a description of your species. What habitat is it found in? What is its geographic range? What, if any, are conservation threats to the species? What data will shed the most light on habitat suitability for this species? 

What core scientific question do you hope to answer about potential future changes in habitat suitability? Don't forget to cite your sources!

Your response here:

### Step 1b: Select study sites
Based on your research and/or range maps you find online, select at least 2 sites where your species occurs. These could be national parks, national forests, national grasslands or other protected areas, or some other area you're interested in. You can access protected area polygons from the [US Geological Survey's Protected Area Database](https://www.usgs.gov/programs/gap-analysis-project/science/pad-us-data-overview), [national grassland units](https://data.fs.usda.gov/geodata/edw/edw_resources/shp/S_USA.NationalGrassland.zip), etc.

When selecting your sites, you might want to look for places that are marginally habitable for this species, since those locations will be most likely to show changes due to climate.

Generate a site map for each location.

In [20]:
### Insert your site maps here:

**Reflect and Respond**: 
Write a site description for each of your sites, or for all of your sites as a group if you have chosen a large number of linked sites. What
differences or trends in habitat suitability over time do you expect to see among your sites?

Your response here:

### Step 1c: Select time periods

In general when studying climate, we are interested in **climate
normals**, which are typically calculated from 30 years of data so that
they reflect the climate as a whole and not a single year which may be
anomalous. So if you are interested in the climate around 2050, you will need to access climate data from 2035-2065.

**Reflect and Respond**: Select at least two 30-year time periods to compare, such as historical and 30 years into the future. These time periods should help you to answer your scientific question.

Your response here:

### Step 1d: Select climate models

There is a great deal of uncertainty among the many global climate
models available. One way to work with the variety is by using an
**ensemble** of models to try to capture that uncertainty. This also
gives you an idea of the range of possible values you might expect! To
be most efficient with your time and computing resources, you can use a
subset of all the climate models available to you. However, for each
scenario, you should attempt to include models that are:

-   Warm and wet
-   Warm and dry
-   Cold and wet
-   Cold and dry

for each of your sites.

To figure out which climate models to use, you will need to access
summary data near your sites for each of the climate models. You can do
this using the [Climate Futures Toolbox Future Climate Scatter
tool](https://climatetoolbox.org/tool/Future-Climate-Scatter). There is
no need to write code to select your climate models, since this choice
is something that requires your judgement and only needs to be done
once.

If your question requires it, you can also choose to include multiple
climate variables, such as temperature and precipitation, and/or
multiple emissions scenarios, such as RCP4.5 and RCP8.5.

**Reflect and respond**: Choose at least 4 climate models that cover the range of possible future climate variability at your sites. Which models did you choose, and how did you make that decision?

Your response here (don't forget to cite the Climate Toolbox): 

## STEP 2: Data access

### Step 2a: Soil data

The [POLARIS dataset](http://hydrology.cee.duke.edu/POLARIS/) is a
convenient way to uniformly access a variety of soil parameters such as
pH and percent clay in the US. It is available for a range of depths (in
cm) and split into 1x1 degree tiles.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Write a <strong>function with a numpy-style docstring</strong> that
will download POLARIS data for a particular location, soil parameter,
and soil depth. Your function should account for the situation where
your site boundary crosses over multiple tiles, and merge the necessary
data together.</p>
<p>Then, use loops to download and organize the rasters you will need to
complete this section. Include soil parameters that will help you to
answer your scientific question. We recommend using a soil depth that
best corresponds with the rooting depth of your species.</p></div></div>

In [21]:
### Download and process soil data

In [22]:
### Visualize the soil data

### Step 2b: Topographic data

Depending on your species habitat needs/environmental parameters, you might be interested in elevation, slope, and/or aspect. You can access reliable elevation data from the [SRTM
dataset](https://www.earthdata.nasa.gov/data/instruments/srtm),
available through the [earthaccess
API](https://earthaccess.readthedocs.io/en/latest/quick-start/). Once you have elevation data, you can calculate slope and aspect.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Write a <strong>function with a numpy-style docstring</strong> that
will download SRTM elevation data for a particular location and
calculate any additional topographic variables you need such as slope or
aspect.</p>
<p>Then, use loops to download and organize the rasters you will need to
complete this section. Include topographic parameters that will help you
to answer your scientific question.</p></div></div>

> **Warning**
>
> Be careful when computing the slope from elevation that the units of
> elevation match the projection units (e.g. meters and meters, not
> meters and degrees). You will need to project the SRTM data to
> complete this calculation correctly.

In [23]:
### Download and process topographic data

In [24]:
### Visualize the topographic data

### Step 2c: Climate model data

You can use MACAv2 data for historical and future climate data. Be sure
to compare at least two 30-year time periods (e.g. historical vs. 10
years in the future) for at least four of the CMIP models. Overall, you
should be downloading at least 8 climate rasters for each of your sites,
for a total of 16. **You will *need* to use loops and/or functions to do
this cleanly!**.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Write a <strong>function with a numpy-style docstring</strong> that
will download MACAv2 data for a particular climate model, emissions
scenario, spatial domain, and time frame. Then, use loops to download
and organize the 16+ rasters you will need to complete this section. The
<a
href="http://thredds.northwestknowledge.net:8080/thredds/reacch_climate_CMIP5_macav2_catalog2.html">MACAv2
dataset is accessible from their Thredds server</a>. Include an
arrangement of sites, models, emissions scenarios, and time periods that
will help you to answer your scientific question.</p></div></div>

In [25]:
### Download climate data

**Reflect and respond**: Make sure to include a description of the climate data and how you selected your models. Include a citation of the MACAv2 data.

Your response here:

## STEP 3: Harmonize data
To use all your environmental and climate data layers together, you need to harmonize the different rasters you've downloaded and processed. 

As a first step, make sure that the grids for all the rasters match each other. Check out the <a href="https://corteva.github.io/rioxarray/stable/examples/reproject_match.html#Reproject-Match"><code>ds.rio.reproject_match()</code> method</a> from <code>rioxarray</code>. Make sure to use the data source that has the highest resolution as a template!</p></div></div>

> **Warning**
>
> If you are reprojecting data (as you need to here), the order of
> operations is important! Recall that reprojecting will typically tilt
> your data, leaving narrow sections of the data at the edge blank.
> However, to reproject efficiently it is best for the raster to be as
> small as possible before performing the operation. We recommend the
> following process:
>
>     1. Crop the data, leaving a buffer around the final boundary
>     2. Reproject to match the template grid (this will also crop any leftovers off the image)

In [26]:
### Align the grids of the different data layers

## STEP 4: Develop a fuzzy logic model

A fuzzy logic model is one that is built on expert knowledge rather than
training data. You may wish to use the
[`scikit-fuzzy`](https://pythonhosted.org/scikit-fuzzy/) library, which
includes many utilities for building this sort of model. In particular,
it contains a number of **membership functions** which can convert your
data into values from 0 to 1 using information such as, for example, the
maximum, minimum, and optimal values for soil pH.

To train a fuzzy logic habitat suitability model:</p>
<pre><code>1. Find the optimal values for your species for each variable you are using (e.g. soil pH, slope, and current annual precipitation). 
2. For each **digital number** in each raster, assign a **continuous** value from 0 to 1 for how close that grid square/pixel is to the optimum range (1 = optimal, 0 = incompatible). 
3. Combine your layers by multiplying them together. This will give you a single suitability number for each grid square.
4. Optionally, you may apply a suitability threshold to make the most suitable areas pop on your map.</code></pre></div></div>

> **Tip**
>
> If you use mathematical operators on a raster in Python, it will
> automatically perform the operation for every number in the raster.
> This type of operation is known as a **vectorized** function. **DO NOT
> DO THIS WITH A LOOP!**. A vectorized function that operates on the
> whole array at once will be much easier and faster.

In [27]:
### Create fuzzy logic model for habitat suitability

## STEP 5: Present your results
Generate some plots that show your key findings of habitat suitability in your study sites across the different time periods and climate models. Don’t forget to interpret your plots!

In [28]:
### Create plots

Interpret your plots here: